# 08 — Production controls: budgets, checkpoints, approval gates

**What you'll learn**

- Cap a run with `shoplab.controls.Budget` — a ceiling on calls and dollars, charged from the loop's `on_step` hook, that raises `BudgetExceeded` the instant it is crossed
- Checkpoint agent state with `Checkpoint.save`/`load` — `{messages, step, meta}` written atomically — and resume a stopped run from where it left off
- Gate the irreversible tools with `require_approval`: wrap `issue_refund` so a denying approver returns a blocked record and no money moves, an approving one lets it through
- Why the WORLD's risky pair (`issue_refund`, `create_replacement`) must never fire unattended, and how a gate makes that a property of the toolset, not a hope about the prompt
- Combine all three into one budgeted, checkpointed, gated triage run

*Time: ~4 min on a first live run; under a minute cached. Cost: ~$0.01. Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

The frozen cell finds or starts the local Phoenix server and traces every LiteLLM call this notebook makes — the handful of triage runs below. Watching a budgeted run stop early, or a gated run make no risky call, is a useful second view of the controls. Optional: skip it and nothing else changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## Three controls a production agent cannot skip

An agent that reads its own way through a task is convenient right up until it is not. Left alone it can loop until the budget is gone, lose everything it had done the moment the process dies, and move real money on a wrong call with nobody watching. None of those is fixed by a better prompt. Each is fixed by a limit the surrounding code enforces, whatever the model decides.

Three controls, one per failure mode. A *budget* caps what a run may spend, in calls or dollars. A *checkpoint* saves state so a crash becomes a pause instead of a loss. An *approval gate* puts a human between the agent and any irreversible act. This chapter builds all three as `shoplab.controls`, then runs them together on the ops desk.

| Control | The failure it removes | The mechanism |
|---|---|---|
| Budget | a run that spends without limit | `on_step` charges every call; `charge` raises `BudgetExceeded` past the cap |
| Checkpoint | a crash that loses the whole run | `{messages, step, meta}` saved atomically; `load`, then resume the loop |
| Approval gate | an irreversible tool firing unattended | `require_approval` wraps the risky `Tool`; a human approves or the call is blocked |

## The agent that moves money

Every agent so far only decided; it printed a verdict and stopped. A production ops-desk agent acts on the verdict — it sends the refund. Same loop from chapter 02, the same nine tools from the WORLD, the same `render_ticket` from chapter 04. The only change is one clause in the system prompt: after deciding, carry the decision out — `issue_refund` when money is due, `create_replacement` for a swap, `escalate` for a human.

That clause is the whole difference between an advisor and something that touches the ledger, and it is exactly where controls stop being optional. The two tools it now reaches for, `issue_refund` and `create_replacement`, are the WORLD's risky pair: irreversible, money-moving, and the object of every guardrail in this chapter and the next. Run the agent once, with none of them, on a vip's opened-item return, and read what it did.

In [ ]:
import json
import shoplab.llm
from shoplab.loop import run_agent
from shoplab.tools import standard_tools, Ledger, run_tool, to_openai_tools
from shoplab.world import load_tickets

ticket = next(t for s in load_tickets().values() for t in s
              if t["ticket_id"] == "TKT-2228")   # CUST-01 (vip), opened, in-window refund

def render_ticket(t):                            # the renderer from chapter 04
    return (f"Ticket {t['ticket_id']} from {t['customer_id']} about order "
            f"{t['order_id']}, sku {t['sku']}, qty {t['qty']}, condition "
            f"{t['item_condition']}, days since delivery {t['days_since_delivery']}, "
            f"photo evidence {t['evidence_photo']}, requested action "
            f"{t['requested_action']}. Customer writes: {t['reason_text']}")

In [ ]:
SYSTEM = ("You are the operations desk agent for Larkspur Outfitters. Look up the "
          "order, the customer, and the one relevant policy, then stop looking "
          "things up; search each policy topic at most once. Then carry out the "
          "decision: call issue_refund when money is due, create_replacement for a "
          "replacement, or escalate for a human review; only then call finish with "
          "decision, policy_id, and refund_usd (number or null). "
          "Decisions follow shop policy, not sympathy.")

ledger = Ledger()
result = run_agent(render_ticket(ticket), standard_tools(ledger), system=SYSTEM, max_steps=8)
print("decision:    ", result.answer)
print("side effects:", ledger.entries)

> **What you should see:** the agent worked the ticket and then *issued a real refund* — the `Ledger` holds one `issue_refund` entry, order ORD-7313 for $129.99 (the full item value a vip's opened, in-window return is owed), with no one asked and nothing in the way. On a correct call that is the job. But the same unattended loop would have fired on a wrong call just as readily, and `issue_refund` does not come back. Everything below makes that fire impossible without a limit, a save point, or a human agreeing to it.

## Budgets: a ceiling the run cannot cross

The loop already takes a `max_steps`, which caps iterations. It does not cap dollars, and iterations are a poor proxy for spend — one step can fan out several tool calls, and a step that calls a strong model costs many times one that does not. A budget counts the things you actually care about: model calls and the money they cost, each with a hard ceiling.

The idea is a running tally with two optional caps. Every model call charges it; the moment either the call count or the accumulated cost crosses its ceiling, `charge` raises `BudgetExceeded` and the run stops. It is deliberately dumb — a counter and two comparisons — because a guardrail you cannot read line by line is not one you should trust with the credit card. Here it is, pasted from `src/shoplab/controls.py`.

In [ ]:
from dataclasses import dataclass

class BudgetExceeded(Exception):
    """Raised by ``Budget.charge`` the moment a call or cost ceiling is crossed."""

# >>> shoplab.controls.Budget
@dataclass
class Budget:
    """A running tally of calls and dollars with optional ceilings; ``charge``
    raises ``BudgetExceeded`` the instant either is crossed. Drive it from the loop
    ``on_step`` hook: ``budget.charge(shoplab.llm.LEDGER[-1]["cost_usd"])``."""
    max_calls: int | None = None
    max_cost_usd: float | None = None
    calls: int = 0
    cost_usd: float = 0.0
    def charge(self, cost=0.0):
        self.calls += 1
        self.cost_usd += cost or 0.0
        if self.max_calls is not None and self.calls > self.max_calls:
            raise BudgetExceeded(f"call budget spent: {self.calls} > {self.max_calls}")
        if self.max_cost_usd is not None and self.cost_usd > self.max_cost_usd:
            raise BudgetExceeded(f"cost budget spent: {self.cost_usd:.6f} > {self.max_cost_usd:.6f}")
        return self
    @property
    def remaining_calls(self):
        return None if self.max_calls is None else self.max_calls - self.calls
    @property
    def remaining_cost(self):
        return None if self.max_cost_usd is None else round(self.max_cost_usd - self.cost_usd, 6)
    def snapshot(self) -> dict:
        return {"calls": self.calls, "cost_usd": round(self.cost_usd, 6),
                "remaining_calls": self.remaining_calls, "remaining_cost": self.remaining_cost}
# <<< shoplab.controls.Budget

That is the whole control, and it is now `shoplab.controls.Budget`. Wiring it to the loop is one line: `run_agent` calls `on_step(step, msg)` after every model call, so an `on_step` that charges the newest `shoplab.llm.LEDGER` row turns the cost ledger the course has kept since chapter 01 into a live meter. Give it a deliberately low ceiling — three calls — and point it at the same ticket, which needs more than three to finish.

In [ ]:
from shoplab.controls import Budget, BudgetExceeded

budget = Budget(max_calls=3)

def charge(step, msg):
    budget.charge(shoplab.llm.LEDGER[-1]["cost_usd"])

try:
    run_agent(render_ticket(ticket), standard_tools(), system=SYSTEM,
              max_steps=8, on_step=charge)
    print("finished within budget")
except BudgetExceeded as stop:
    print("halted:", stop)
print("snapshot:", budget.snapshot())

> **What you should see:** the run does not finish. On the fourth charge the tally crosses the ceiling of three and `charge` raises `BudgetExceeded` — the message reads `call budget spent: 4 > 3` — so the loop stops mid-ticket, before any `finish`. The `snapshot` shows `remaining_calls` gone negative by exactly one: the tripping call is counted, then rejected. Swap `max_calls=3` for a `max_cost_usd` and the same tripwire fires on dollars instead. The cap is a guarantee enforced by code, not a hope pinned on the prompt — the model never gets a say in whether to honor it.

## Checkpoints: state you can resume from

A budget stops a runaway; it does nothing for a run that dies halfway through honest work — a killed process, a timed-out call, a machine that reboots. Without saved state the agent starts the ticket over, re-paying for every lookup it already did. A checkpoint is the fix: write the run's state to disk often enough that a crash costs one step, not the whole run.

The state worth saving is small — the running `messages`, the `step` reached, and a slot for bookkeeping. The one subtlety is the write itself: a process that dies *while saving* must not leave a half-written file that loads as garbage. So `save` writes to a temporary sibling and atomically renames it into place; a reader sees either the old checkpoint or the new one, never a torn one. Pasted from the package, the save half is the chapter's second sentinel region. This is LangGraph's persistence idea in miniature — short-term memory an agent can be restored from ([LangChain docs](https://docs.langchain.com/oss/python/langgraph/persistence)).

In [ ]:
import json
from pathlib import Path

class Checkpoint:
    """Save and restore agent state as JSON so a run can resume after a crash."""

    # >>> shoplab.controls.Checkpoint.save
    @staticmethod
    def save(state: dict, path) -> Path:
        """Atomically write ``{messages, step, meta}`` as JSON; return the Path.

        Dumps to a sibling ``.tmp`` then ``replace``s it into place, so a reader
        never sees a half-written file even if the process dies mid-write."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {"messages": state.get("messages", []),
                   "step": state.get("step", 0), "meta": state.get("meta", {})}
        tmp = path.with_suffix(path.suffix + ".tmp")
        tmp.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        tmp.replace(path)
        return path
    # <<< shoplab.controls.Checkpoint.save

    @staticmethod
    def load(path) -> dict:
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"no checkpoint at {path} -- nothing to resume from")
        return json.loads(path.read_text(encoding="utf-8"))

`save` and `load` are now `shoplab.controls.Checkpoint`. To see them earn their keep, cut a run short — two steps, as if the process were killed — save what it had, then bring it back and resume the loop from there. The restored state is just a dict of messages and a step number, but that is the entire working memory of the loop: enough to continue from where it stopped rather than from the top.

In [ ]:
from shoplab.controls import Checkpoint

led = Ledger()
tools = standard_tools(led)
partial = run_agent(render_ticket(ticket), tools, system=SYSTEM, max_steps=2)
state = {"messages": partial.messages, "step": partial.steps,
         "meta": {"ticket": ticket["ticket_id"]}}
Checkpoint.save(state, "artifacts/checkpoints/TKT-2228.json")

restored = Checkpoint.load("artifacts/checkpoints/TKT-2228.json")
print("stopped after", partial.stop_reason, "at step", restored["step"],
      "with", len(restored["messages"]), "messages")

In [ ]:
def resume(state, tools, *, model=None, max_steps=8):   # the continue-the-loop half of run_agent
    messages = list(state["messages"])
    for step in range(state["step"] + 1, state["step"] + 1 + max_steps):
        msg = shoplab.llm.complete(messages, tools=to_openai_tools(tools),
                                   model=model).choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls or []:
            out = run_tool(tools, tc.function.name, json.loads(tc.function.arguments or "{}"))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out[:2000]})
            if tc.function.name == "finish":
                return {"answer": json.loads(tc.function.arguments), "step": step}
    return {"answer": None, "step": step}

done = resume(restored, tools)
print("resumed ->", done["answer"], "at step", done["step"])
print("side effects:", led.entries)

> **What you should see:** the resumed run picks up at step 3, not step 1. The first two steps' lookups already sit in `messages`, so `resume` spends no calls redoing them; it runs the remaining steps and reaches `finish` with the same `approve_refund` for $129.99, and the refund lands in the ledger. `resume` is deliberately just the body of `run_agent` starting from a saved message list — a checkpoint is worth exactly as much as your ability to feed it back into the loop. Real deployments checkpoint every step (exercise 2); two was enough to make the point. The files live under `artifacts/checkpoints/`.

## Approval gates: a human on the irreversible acts

A budget and a checkpoint bound *how much* and *how far*; neither asks whether a particular act should happen at all. For the irreversible ones — sending money, shipping a unit — the right control is a human who says yes first. `require_approval` builds that gate without touching the tool: it wraps a `Tool` in a new `Tool` with the same name, schema, and `risky` flag, whose function first calls an `approver(name, args)`. Approve, and the real tool runs; deny, and it returns `{"blocked": True, ...}` and never fires. The agent cannot tell the wrapped tool from the original by its schema, and it cannot route around a tool that simply refuses to act.

The approver is any `(name, args) -> bool`. At a console it prompts a person; a test passes a lambda. This is the same move as LangGraph's human-in-the-loop interrupt — pause on the risky step and wait for external input before proceeding ([LangChain docs](https://docs.langchain.com/oss/python/langgraph/interrupts)) — done at the granularity of a single tool. Wrap the risky pair with a denying approver, then an approving one, and send the same ticket through both.

In [ ]:
from shoplab.controls import require_approval

locked = require_approval(standard_tools()["issue_refund"], lambda name, args: False)
print("gate denied ->", locked.fn(order_id=ticket["order_id"], amount_usd=129.99, reason="demo"))

def gated_run(approver):
    led = Ledger()
    tools = standard_tools(led)
    for name in ("issue_refund", "create_replacement"):
        tools[name] = require_approval(tools[name], approver)
    answer = run_agent(render_ticket(ticket), tools, system=SYSTEM, max_steps=8).answer
    return led.entries, answer

print("auto-DENY    ->", gated_run(lambda name, args: False))
print("auto-APPROVE ->", gated_run(lambda name, args: True))

> **What you should see:** the isolated gate, called directly with a denying approver, returns `{'blocked': True, 'reason': 'approval denied'}` and writes nothing. Then end to end: under `auto-DENY` the ledger is empty — the agent tried to refund, the gate refused, no money moved — and under `auto-APPROVE` the same run puts the $129.99 `issue_refund` in the ledger. Same agent, same ticket, same prompt; the only variable is the human's answer. That is the property you want: the risky pair cannot fire unattended, because firing now requires a `True` that only a human — or a policy you write, exercise 3 — can supply.

## All three at once

The controls compose because they hook different seams. The budget rides `on_step`; the gate wraps the tools; the checkpoint reads the result. Nothing about one knows or cares about the others, so a production triage wears all three at once: a comfortable ceiling it will not cross, an approver on the money-movers, and a saved state when it is done. This is the shape a real ops-desk run would take.

In [ ]:
led = Ledger()
tools = standard_tools(led)
for name in ("issue_refund", "create_replacement"):
    tools[name] = require_approval(tools[name], lambda name, args: True)

budget = Budget(max_calls=10, max_cost_usd=0.05)
def guard(step, msg):
    budget.charge(shoplab.llm.LEDGER[-1]["cost_usd"])

run = run_agent(render_ticket(ticket), tools, system=SYSTEM, max_steps=8, on_step=guard)
Checkpoint.save({"messages": run.messages, "step": run.steps,
                 "meta": {"decision": run.answer}}, "artifacts/checkpoints/combined.json")
print("decision:", run.answer)
print("budget:  ", budget.snapshot())
print("moved:   ", led.entries)

> **What you should see:** one clean run that stays inside every control. The decision is `approve_refund` for $129.99; the `budget` snapshot shows the run finished under both ceilings, with calls and dollars to spare (`remaining_calls` and `remaining_cost` both positive); the money moved only because the approver said yes; and the final state is on disk under `artifacts/checkpoints/`. Take one control away and you reintroduce exactly one failure mode from the opening table — that is the argument for keeping all three.

## Recap

| Concept | One-liner |
|---|---|
| Three controls | budget, checkpoint, approval gate — one per production failure mode, each enforced by code around the model. |
| `Budget` | a tally with call and cost ceilings; `charge` raises `BudgetExceeded` the instant either is crossed. |
| `on_step` meter | charging `shoplab.llm.LEDGER[-1]["cost_usd"]` each step turns the loop's hook into a live spend limit. |
| `BudgetExceeded` | stops the run mid-loop; the cap is a guarantee, not a hope pinned on the prompt. |
| `Checkpoint.save` | writes `{messages, step, meta}` atomically (temp file, then rename), so a crash never yields a torn file. |
| Resume | a checkpoint is worth what you can feed back: continue `run_agent`'s loop from the saved messages. |
| `require_approval` | wraps a risky `Tool` in a gate; deny returns a blocked record, approve runs the real tool. |
| Risky pair | `issue_refund` and `create_replacement` never fire unattended — approval is a property of the toolset. |
| Composition | budget on `on_step`, gate on the tools, checkpoint on the result — independent seams, stacked freely. |

## Exercises

1. Add a wall-clock ceiling. Extend `Budget` with a `max_wall_seconds` and a `started_at` timestamp so `charge` also trips when a run has simply taken too long — the control that catches a stuck tool call a call-count cap would miss. Drive it from `on_step` on a normal ticket and confirm a tight limit (say one second) halts the run with `BudgetExceeded`.
2. Checkpoint every step and resume from an arbitrary one. Change the first run's `on_step` to `Checkpoint.save` the state after each step under numbered files (`step-01.json`, `step-02.json`, ...), then load `step-02.json` and resume — do you land on the same decision as an uninterrupted run? What does resuming from `step-01.json` versus a later one cost in calls?
3. Make the approver policy-aware. Write an `approver(name, args)` that auto-approves an `issue_refund` under $50 and denies the larger ones (returning `True`/`False` without prompting, standing in for a human queue). Run it over the dev tickets and count how many refunds would clear automatically — the small-dollar, high-volume tail — versus how many a person would actually have to look at.

**Next up:** chapter 09 turns the inputs hostile. The gate that looks optional here becomes mandatory: once product reviews and customer emails carry prompt injections aimed at `issue_refund`, an agent without a control on its risky tools is one crafted sentence away from paying out.